In [19]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Import 

In [ ]:
from models import Simple_Model
from utils import EarlyStopping, get_device

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

# Parameters

In [ ]:
batch_size       = 64
epoch            = 100
hidden_size      = 128
num_output       = 1
num_hidden_layer = 3
dropout          = 0.3
patience         = 10
test_size        = 0.2
lr = 0.001

# Sample data

## Option 1 : Get raw data (random)

In [22]:
features_num = 10
record_cnt = 100

X = torch.randint( 0, 10, (record_cnt, features_num))
y = torch.randint( 0, 10, (record_cnt,))

## Option 2 :  Get raw data from sklearn dataset

In [23]:
# X,y = load_diabetes(return_X_y = True)

# Data Processing

## split data into training and testing dataset

In [24]:
y = y.reshape( len(y), 1)

In [25]:
X_train , X_test , y_train, y_test = train_test_split(X, y, test_size = test_size)

## Data scaling

In [ ]:
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

preprocess_config = {
    "x_mean": X_scaler.mean_.tolist(),
    "x_scale": X_scaler.scale_.tolist(),
    "y_mean": y_scaler.mean_.tolist(),
    "y_scale": y_scaler.scale_.tolist(),
    "task_type": "regression",
}

## Transform to Tensor dataset

In [27]:
X_train_tensor = torch.from_numpy( X_train_scaled).float()
X_test_tensor = torch.from_numpy( X_test_scaled).float()

y_train_tensor = torch.from_numpy( y_train_scaled).float()
y_test_tensor = torch.from_numpy( y_test_scaled).float()

In [29]:
train_dataset = TensorDataset( X_train_tensor, y_train_tensor)
test_datset = TensorDataset( X_test_tensor, y_test_tensor)

In [30]:
train_loader = DataLoader( train_dataset, batch_size = batch_size, shuffle = True )
test_loader = DataLoader( test_datset, batch_size = batch_size, shuffle = True )

# Prepare model

In [ ]:
device = get_device()

In [32]:
num_cols = X_train.shape[1]

In [ ]:
model = Simple_Model(
    input_size = num_cols,
    hidden_size = hidden_size,
    num_output = num_output,
    num_hidden_layer = num_hidden_layer,
    dropout = dropout 
 )

_ = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
criterion = nn.MSELoss()

model_config = {
    "input_size": num_cols,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_hidden_layer": num_hidden_layer,
    "dropout": dropout,
}

In [ ]:
early_stopping = EarlyStopping(
    patience = patience,
    path='../checkpoints/simple_checkpoint.pt',
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
    },
)

# Loop epochs and train model

In [36]:
for epoch in range(epoch):
    
    # ------------------- Training -------------------
    model.train()
    train_loss = 0.0

    
    for batch_x, batch_y in train_loader:
        
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)

        loss = criterion(output.squeeze(), batch_y)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ------------------- Validation -------------------

    model.eval()
    val_loss = 0.0

    all_preds = []
    all_actual = []

    with torch.no_grad():
        
        for batch_x, batch_y in test_loader:

            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            predit = model(batch_x).squeeze()
            actual = batch_y.squeeze()


            loss = criterion(predit, batch_y)
            val_loss += loss.item()

            
            all_preds.append(predit)
            all_actual.append(actual)


    # ------------------- Evaluation -------------------    
    
    # Average loss
    avg_val_loss = val_loss / len(test_loader)          

    # R2
    all_preds   = torch.cat(all_preds)
    all_actual = torch.cat(all_actual)

    ss_res = ((all_actual - all_preds) ** 2).sum()
    ss_tot = ((all_actual - all_actual.mean()) ** 2).sum()
    r2 = (1 - ss_res / ss_tot).item()
            

    print(f'Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | R²: {r2:.4f}')
    
    # ==================== Early Stopping Check ====================
    early_stopping(avg_val_loss, model)
    
    if early_stopping.early_stop:
        print("Early stopping triggered! Training stopped.")
        break

print("Training finished!")
    

C:\Users\ada_lau\Documents\Work\Learn\pytorch-case\myenv\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\ada_lau\Documents\Work\Learn\pytorch-case\myenv\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([16, 1])) that is different to the input size (torch.Size([16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\ada_lau\Documents\Work\Learn\pytorch-case\myenv\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to

Epoch   1 | Train Loss: 0.992154 | Val Loss: 0.876962 | R²: -0.0894
Epoch   2 | Train Loss: 1.112313 | Val Loss: 0.885125 | R²: -0.0995
Epoch   3 | Train Loss: 0.987043 | Val Loss: 0.876196 | R²: -0.0883
Epoch   4 | Train Loss: 0.897586 | Val Loss: 0.873938 | R²: -0.0853
Epoch   5 | Train Loss: 1.030056 | Val Loss: 0.875388 | R²: -0.0871
Epoch   6 | Train Loss: 0.915831 | Val Loss: 0.878866 | R²: -0.0914
Epoch   7 | Train Loss: 0.997940 | Val Loss: 0.881907 | R²: -0.0951
Epoch   8 | Train Loss: 1.010566 | Val Loss: 0.886238 | R²: -0.1006
Epoch   9 | Train Loss: 1.014067 | Val Loss: 0.887807 | R²: -0.1026
Epoch  10 | Train Loss: 0.974861 | Val Loss: 0.885145 | R²: -0.0994
Epoch  11 | Train Loss: 1.057660 | Val Loss: 0.879562 | R²: -0.0929
Epoch  12 | Train Loss: 1.120351 | Val Loss: 0.874157 | R²: -0.0865
Epoch  13 | Train Loss: 0.979931 | Val Loss: 0.867462 | R²: -0.0779
Epoch  14 | Train Loss: 1.047071 | Val Loss: 0.861835 | R²: -0.0707
Epoch  15 | Train Loss: 1.043145 | Val Loss: 0.8